# DataDriven_Soccer_Scouting

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from tensorflow.keras import layers, models

import warnings
warnings.filterwarnings("ignore")

## Introduction and EDA

In [3]:
df = pd.read_csv("merged_data.csv")
# Convert 'season' to a more readable format (e.g., '2020 - 2021')
df['season'] = df['season'].astype(str).str.zfill(4)
df['season'] = "20" + df['season'].str[:2] + " - 20" + df['season'].str[2:]

display(df.head())
print('\nRows: {}, Columns: {}'.format(df.shape[0], df.shape[1]))

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Touches_Att 3rd,Touches_Att Pen,Touches_Def 3rd,Touches_Def Pen,Touches_Live,Touches_Mid 3rd,Touches_Touches,Height,Weight,Preferred foot
0,ENG-Premier League,2024 - 2025,Arsenal,Ben White,ENG,DF,26,1997,17,13,...,238,28,214,55,813,367,813,NaN,NaN,NaN
1,ENG-Premier League,2024 - 2025,Arsenal,Bukayo Saka,ENG,"FW,MF",22,2001,25,20,...,715,162,57,6,978,216,979,178.0,65.0,Left
2,ENG-Premier League,2024 - 2025,Arsenal,David Raya,ESP,GK,28,1995,38,38,...,0,0,1388,880,1480,92,1480,183.0,75.0,Right
3,ENG-Premier League,2024 - 2025,Arsenal,Declan Rice,ENG,MF,25,1999,35,33,...,733,81,325,82,1948,909,1948,185.0,80.0,Right
4,ENG-Premier League,2024 - 2025,Arsenal,Gabriel Magalhães,BRA,DF,26,1997,28,28,...,225,40,672,189,1873,985,1873,NaN,NaN,NaN



Rows: 4807, Columns: 117


In [4]:
#Removing rows with null values and Goalkeepers (GK)
not_null_df = df.dropna()
not_null_df = not_null_df[not_null_df['pos'] != 'GK']
print('data without null and GKs\n- rows: {} \n- columns: {}'.format(not_null_df.shape[0], not_null_df.shape[1]))
df_backup = df.copy()
df = not_null_df.copy()

data without null and GKs
- rows: 3396 
- columns: 117


In [5]:
# select all not numeric columns
not_numeric_cols = df.select_dtypes(exclude=[np.number]).columns
display(df[not_numeric_cols].head())

,league,season,team,player,nation,pos,Preferred foot
1,ENG-Premier League,2024 - 2025,Arsenal,Bukayo Saka,ENG,"FW,MF",Left
3,ENG-Premier League,2024 - 2025,Arsenal,Declan Rice,ENG,MF,Right
5,ENG-Premier League,2024 - 2025,Arsenal,Gabriel Martinelli,BRA,"FW,MF",Right
6,ENG-Premier League,2024 - 2025,Arsenal,Jakub Kiwior,POL,DF,Left
7,ENG-Premier League,2024 - 2025,Arsenal,Jurriën Timber,NED,DF,Right


In [6]:
# Convert 'Preferred foot' to binary: 1 for 'Right', 0 for 'Left'
df['right_foot'] = df['Preferred foot'].map({'Right': 1, 'Left': 0})
df.drop('Preferred foot', axis=1, inplace=True)

# Identify info columns and create a separate DataFrame for them
info_columns = ['league', 'season', 'team', 'player', 'nation', 'pos', 'born', 'age']
df_info = df[[col for col in df.columns if col.lower() in info_columns]].copy()
numeric_features = df.select_dtypes(include=[np.number]).columns
features = [col for col in numeric_features if col.lower() not in info_columns]

X = df[features].copy()

print(f"Matrix 'X' shape: {X.shape[0]} players x {X.shape[1]} statistics")

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Matrix 'X' shape: 3396 players x 109 statistics


## Similarity Search

### PCA

In [7]:
# ==========================================
# 1. ADDESTRAMENTO DELLA PCA (Dimensionality Reduction)
# ==========================================
print("--- Addestramento Modello Baseline: PCA ---")

# Scegliamo 10 componenti principali (di solito bastano per trattenere l'80%+ dell'informazione)
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_scaled)

# Controlliamo quanta "informazione" (varianza) abbiamo conservato comprimendo i dati
varianza_totale = sum(pca.explained_variance_ratio_) * 100
print(f"Varianza spiegata con 10 componenti: {varianza_totale:.2f}%")

# ==========================================
# 2. CREAZIONE DELLO "SPAZIO LATENTE"
# ==========================================
# Creiamo un DataFrame con le nuove 10 coordinate matematiche
colonne_pca = [f'PC{i+1}' for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=colonne_pca)

# Uniamo queste coordinate alle informazioni di testo (nomi, squadre, ruoli) salvate prima
# Reset dell'indice per essere sicuri che si allineino perfettamente
df_info_reset = df_info.reset_index(drop=True)
df_latent = pd.concat([df_info_reset, df_pca], axis=1)

# ==========================================
# 3. MOTORE DI RICERCA (Similarity Search)
# ==========================================
def trova_simili_pca(nome_giocatore, df_latente, top_n=5):
    """
    Trova i giocatori più simili usando la Cosine Similarity sulle componenti PCA.
    """
    # Cerca il giocatore ignorando maiuscole/minuscole
    giocatore_idx = df_latente[df_latente['player'].str.lower() == nome_giocatore.lower()].index
    
    if len(giocatore_idx) == 0:
        return f"Errore: Giocatore '{nome_giocatore}' non trovato nel dataset. Prova a controllare come è scritto."
    
    # Se ce n'è più di uno (es. omonimi o stagioni diverse), prendiamo il primo
    idx = giocatore_idx[0] 
    giocatore_reale = df_latente.loc[idx, 'player']
    squadra_reale = df_latente.loc[idx, 'team']
    print(f"\nRicerca cloni per: {giocatore_reale} ({squadra_reale})")
    
    # Isoliamo il "DNA" (vettore a 10 dimensioni) del nostro giocatore target
    giocatore_vettore = df_latente.loc[idx, colonne_pca].values.reshape(1, -1)
    
    # Estraiamo il "DNA" di tutti i giocatori del dataset
    tutti_i_vettori = df_latente[colonne_pca].values
    
    # Calcoliamo la Similarità Coseno (1.0 = identico, 0.0 = totalmente diverso)
    similarita = cosine_similarity(giocatore_vettore, tutti_i_vettori)[0]
    
    # Aggiungiamo il punteggio al dataframe
    df_latente_temp = df_latente.copy()
    df_latente_temp['Similarity_Score'] = similarita
    
    # Ordiniamo dal più simile al meno simile
    # Saltiamo il primo risultato (che sarà il giocatore stesso con score 1.0)
    simili = df_latente_temp.sort_values(by='Similarity_Score', ascending=False).iloc[1:top_n+1]
    
    # Formattiamo lo score in percentuale per renderlo più leggibile
    simili['Match %'] = (simili['Similarity_Score'] * 100).round(1).astype(str) + '%'
    # Mostriamo solo le colonne che ci interessano
    return simili[['player', 'age', 'team', 'pos', 'league', 'season', 'Match %']]

# ==========================================
# 4. TEST DELLO SCOUTING
# ==========================================
target = "Kevin De Bruyne" 
risultati = trova_simili_pca(target, df_latent, top_n=5)
display(risultati)

--- Addestramento Modello Baseline: PCA ---
Varianza spiegata con 10 componenti: 82.22%

Ricerca cloni per: Kevin De Bruyne (Manchester City)


,player,age,team,pos,league,season,Match %
1994,Jonas Hofmann,31,Leverkusen,MF,GER-Bundesliga,2023 - 2024,94.6%
1365,Kevin De Bruyne,32,Manchester City,MF,ENG-Premier League,2023 - 2024,93.7%
732,Julian Brandt,28,Dortmund,MF,GER-Bundesliga,2024 - 2025,91.2%
3324,Piotr Zieliński,28,Napoli,MF,ITA-Serie A,2022 - 2023,90.9%
1041,Federico Dimarco,26,Inter,"DF,FW",ITA-Serie A,2024 - 2025,89.5%


### Deep Autoencoder

In [8]:
print("--- Costruzione Deep Autoencoder (TensorFlow/Keras) ---")

# Assumiamo che X_scaled sia il tuo array numpy scalato (es. con StandardScaler)
input_dim = X_scaled.shape[1] 

# ==========================================
# 1. COSTRUZIONE DELL'ARCHITETTURA (Lego)
# ==========================================
# Input
input_layer = layers.Input(shape=(input_dim,))

# ENCODER (Compressione con attivazione Tanh)
encoded = layers.Dense(64, activation='tanh')(input_layer)
encoded = layers.Dense(32, activation='tanh')(encoded)

# BOTTLENECK (Il nostro Spazio Latente a 10 dimensioni)
bottleneck_layer = layers.Dense(10, activation='tanh', name='dna_bottleneck')(encoded)

# DECODER (Ricostruzione)
decoded = layers.Dense(32, activation='tanh')(bottleneck_layer)
decoded = layers.Dense(64, activation='tanh')(decoded)

# Output finale (Torna al numero di colonne originali, lineare)
output_layer = layers.Dense(input_dim, activation='linear')(decoded)

# ==========================================
# 2. COMPILAZIONE E ADDESTRAMENTO
# ==========================================
# Uniamo l'input all'output per creare il modello completo
autoencoder = models.Model(inputs=input_layer, outputs=output_layer)

# Usiamo 'adam' (ottimo e veloce in Keras) e l'errore quadratico medio
autoencoder.compile(optimizer='adam', loss='mse')

print("\nAddestramento in corso...")
# Addestriamo la rete (verbose=1 mostra una comoda barra di caricamento)
autoencoder.fit(X_scaled, X_scaled, epochs=100, batch_size=32, verbose=1)
print("Addestramento completato!")

# ==========================================
# 3. ESTRAZIONE DELLO SPAZIO LATENTE
# ==========================================
print("\nEstrazione del DNA a 10 dimensioni in corso...")

# Creiamo un SECONDO modello che si ferma al bottleneck
encoder = models.Model(inputs=input_layer, outputs=bottleneck_layer)

# Keras fa tutto il calcolo matriciale per noi con una sola riga!
bottleneck = encoder.predict(X_scaled)

# ==========================================
# 4. MOTORE DI RICERCA (Similarity Search)
# ==========================================
# Creiamo il DataFrame con le 10 nuove coordinate estratte dall'IA
colonne_ae = [f'AE_{i+1}' for i in range(10)]
df_ae = pd.DataFrame(bottleneck, columns=colonne_ae)

# Uniamo l'anagrafica dei giocatori al loro nuovo "DNA"
df_latent_ae = pd.concat([df_info_reset, df_ae], axis=1)

def trova_simili_ae(nome_giocatore, df_latente, top_n=5):
    # Cerchiamo l'indice del giocatore
    giocatore_idx = df_latente[df_latente['player'].str.lower() == nome_giocatore.lower()].index
    if len(giocatore_idx) == 0:
        return f"Giocatore non trovato nel database."
    
    idx = giocatore_idx[0] 
    giocatore_reale = df_latente.loc[idx, 'player']
    squadra_reale = df_latente.loc[idx, 'team']
    
    # Se la colonna 'season' esiste, stampiamola per maggiore chiarezza
    stagione = df_latente.loc[idx, 'season'] if 'season' in df_latente.columns else "N/A"
    print(f"\n[AUTOENCODER] Ricerca cloni per: {giocatore_reale} ({squadra_reale} - {stagione})")
    
    # Estraiamo i 10 numeri del giocatore cercato
    giocatore_vettore = df_latente.loc[idx, colonne_ae].values.reshape(1, -1)
    # Estraiamo i 10 numeri di TUTTI i giocatori
    tutti_i_vettori = df_latente[colonne_ae].values
    
    # Calcoliamo la Similarità del Coseno matematica
    similarita = cosine_similarity(giocatore_vettore, tutti_i_vettori)[0]
    
    # Assegniamo i punteggi
    df_latente_temp = df_latente.copy()
    df_latente_temp['Similarity_Score'] = similarita
    
    # Ordiniamo dal più simile al meno simile, escludendo la prima riga (il giocatore stesso a 100%)
    simili = df_latente_temp.sort_values(by='Similarity_Score', ascending=False).iloc[1:top_n+1].copy()
    
    # Formattiamo la colonna in percentuale per renderla bella da leggere
    simili['Match %'] = (simili['Similarity_Score'] * 100).round(1).astype(str) + '%'
    
    # Selezioniamo le colonne finali da mostrare (aggiungi pure quelle che preferisci)
    colonne_output = ['player', 'age', 'team', 'pos', 'league']
    if 'season' in simili.columns:
        colonne_output.append('season')
    colonne_output.append('Match %')
        
    return simili[colonne_output]

# ==========================================
# 5. TESTIAMO IL MODELLO
# ==========================================
target = "Alessandro Bastoni" 
risultati_ae = trova_simili_ae(target, df_latent_ae, top_n=5)
display(risultati_ae)

--- Costruzione Deep Autoencoder (TensorFlow/Keras) ---

Addestramento in corso...
Epoch 1/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.5815
Epoch 2/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.3630
Epoch 3/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3057
Epoch 4/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.2706
Epoch 5/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2496
Epoch 6/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2352
Epoch 7/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2259
Epoch 8/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2179
Epoch 9/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2123
Epoch 10/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2082
Epoch 11/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2043
Epoch 12/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2009
Epoch 13/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1989
Epoch 14/100
107/107 ━━━━━━━━

,player,age,team,pos,league,season,Match %
2612,Jules Koundé,23,Barcelona,DF,ESP-La Liga,2022 - 2023,91.8%
3361,Rogério,24,Sassuolo,DF,ITA-Serie A,2022 - 2023,88.7%
2805,Facundo Medina,23,Lens,DF,FRA-Ligue 1,2022 - 2023,88.2%
2191,Alessandro Bastoni,24,Inter,DF,ITA-Serie A,2023 - 2024,88.1%
1366,Kyle Walker,33,Manchester City,DF,ENG-Premier League,2023 - 2024,87.1%


In [9]:
target = ["Kevin De Bruyne", "Alessandro Bastoni", "Manuel Locatelli", "Nicolò Barella", "Federico Chiesa", "Riccardo Orsolini"]
for giocatore in target:
    risultati_ae = trova_simili_ae(giocatore, df_latent_ae, top_n=5)
    display(risultati_ae)


[AUTOENCODER] Ricerca cloni per: Kevin De Bruyne (Manchester City - 2024 - 2025)


,player,age,team,pos,league,season,Match %
3324,Piotr Zieliński,28,Napoli,MF,ITA-Serie A,2022 - 2023,90.9%
1365,Kevin De Bruyne,32,Manchester City,MF,ENG-Premier League,2023 - 2024,89.8%
732,Julian Brandt,28,Dortmund,MF,GER-Bundesliga,2024 - 2025,87.3%
460,Alex Baena,23,Villarreal,"MF,FW",ESP-La Liga,2024 - 2025,86.8%
1994,Jonas Hofmann,31,Leverkusen,MF,GER-Bundesliga,2023 - 2024,86.7%



[AUTOENCODER] Ricerca cloni per: Alessandro Bastoni (Inter - 2024 - 2025)


,player,age,team,pos,league,season,Match %
2612,Jules Koundé,23,Barcelona,DF,ESP-La Liga,2022 - 2023,91.8%
3361,Rogério,24,Sassuolo,DF,ITA-Serie A,2022 - 2023,88.7%
2805,Facundo Medina,23,Lens,DF,FRA-Ligue 1,2022 - 2023,88.2%
2191,Alessandro Bastoni,24,Inter,DF,ITA-Serie A,2023 - 2024,88.1%
1366,Kyle Walker,33,Manchester City,DF,ENG-Premier League,2023 - 2024,87.1%



[AUTOENCODER] Ricerca cloni per: Manuel Locatelli (Juventus - 2024 - 2025)


...

,player,age,team,pos,league,season,Match %
1488,Rodrigo De Paul,29,Atlético Madrid,MF,ESP-La Liga,2023 - 2024,93.2%
1210,Oleksandr Zinchenko,26,Arsenal,DF,ENG-Premier League,2023 - 2024,90.3%
1044,Henrikh Mkhitaryan,35,Inter,MF,ITA-Serie A,2024 - 2025,89.9%
2601,Rodrigo De Paul,28,Atlético Madrid,MF,ESP-La Liga,2022 - 2023,89.4%
2877,Jordan Ferri,30,Montpellier,MF,FRA-Ligue 1,2022 - 2023,89.3%



[AUTOENCODER] Ricerca cloni per: Federico Chiesa (Juventus - 2023 - 2024)


,player,age,team,pos,league,season,Match %
1173,Florian Thauvin,31,Udinese,"FW,MF",ITA-Serie A,2024 - 2025,94.8%
1517,Jonathan Bamba,27,Celta Vigo,"MF,FW",ESP-La Liga,2023 - 2024,91.6%
1546,Bryan Zaragoza,21,Granada,"MF,FW",ESP-La Liga,2023 - 2024,91.3%
428,Takefusa Kubo,23,Real Sociedad,"MF,FW",ESP-La Liga,2024 - 2025,91.0%
1609,Takefusa Kubo,22,Real Sociedad,"FW,MF",ESP-La Liga,2023 - 2024,90.2%



[AUTOENCODER] Ricerca cloni per: Riccardo Orsolini (Bologna - 2024 - 2025)


,player,age,team,pos,league,season,Match %
1736,Georges Mikautadze,22,Metz,FW,FRA-Ligue 1,2023 - 2024,92.5%
1711,Alexandre Lacazette,32,Lyon,FW,FRA-Ligue 1,2023 - 2024,89.5%
2109,Riccardo Orsolini,26,Bologna,"FW,MF",ITA-Serie A,2023 - 2024,88.0%
813,Shuto Machino,24,Holstein Kiel,"FW,MF",GER-Bundesliga,2024 - 2025,86.1%
2005,Jonathan Burkardt,23,Mainz 05,"FW,MF",GER-Bundesliga,2023 - 2024,85.3%


## Anomaly Detection

### SVM

### Autoencoder Reconstruction Error